# 02 — Enriquecimiento de IOCs con Python

Reproducimos la lógica de:
- `backend/app/services/ioc_enrichment.py` (`detect_ioc_type`, `simulate_*`)
- `backend/app/services/ioc_service.py` (score, veredicto, recomendación)

**Librerías:** `re`, `hashlib` (stdlib) + `pandas` / `matplotlib` para métricas.

## 1. Análisis del código: detección de tipo IOC

```python
import hashlib, re
IP_PATTERN = re.compile(r"^(?:(?:25[0-5]|2[0-4]\d|[01]?\d?\d)(?:\.|$)){4}$")
HASH_PATTERN = re.compile(r"^[a-fA-F0-9]{32,64}$")
```

- **IP:** regex de IPv4 válida
- **Hash:** longitud 32 → MD5, 40 → SHA1, resto → SHA256
- **URL:** empieza por `http`

Los simuladores usan **hash determinista** del valor: misma IP → mismo score (útil en demos y tests).

In [ ]:
import hashlib
import re
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

# --- Código equivalente a ioc_enrichment.py ---
IP_PATTERN = re.compile(
    r"^(?:(?:25[0-5]|2[0-4]\d|[01]?\d?\d)(?:\.|$)){4}$"
)
HASH_PATTERN = re.compile(r"^[a-fA-F0-9]{32,64}$")


def detect_ioc_type(value: str) -> str:
    value = value.strip()
    if IP_PATTERN.match(value):
        return "ip"
    if HASH_PATTERN.match(value):
        if len(value) == 32:
            return "md5"
        if len(value) == 40:
            return "sha1"
        return "sha256"
    if value.startswith("http"):
        return "url"
    return "unknown"


def simulate_abuseipdb(ip: str) -> dict:
    score = int(hashlib.md5(ip.encode()).hexdigest(), 16) % 101
    return {
        "source": "AbuseIPDB",
        "mode": "simulated",
        "abuse_confidence_score": score,
        "country": "Unknown",
        "total_reports": score // 10,
    }


def simulate_virustotal(value: str, ioc_type: str) -> dict:
    seed = int(hashlib.sha256(value.encode()).hexdigest(), 16)
    malicious = seed % 70
    suspicious = (seed // 7) % 20
    harmless = max(70 - malicious, 0)
    undetected = max(100 - malicious - suspicious - harmless, 0)
    return {
        "source": "VirusTotal",
        "mode": "simulated",
        "malicious": malicious,
        "suspicious": suspicious,
        "harmless": harmless,
        "undetected": undetected,
    }


def compute_verdict(risk_score: int) -> str:
    if risk_score >= 75:
        return "malicious"
    if risk_score >= 40:
        return "suspicious"
    return "clean"


def vt_risk_score(vt: dict) -> int:
    total = vt["malicious"] + vt["suspicious"] + vt["harmless"] + vt["undetected"]
    if total <= 0:
        return 0
    return min(100, int((vt["malicious"] * 100 + vt["suspicious"] * 50) / total))


def enrich_ioc(value: str) -> dict:
    """Equivalente simplificado de ioc_service.enrich_ioc (modo simulado)."""
    value = value.strip()
    ioc_type = detect_ioc_type(value)
    vt = simulate_virustotal(value, ioc_type)
    risk = vt_risk_score(vt)
    abuse = None
    if ioc_type == "ip":
        abuse = simulate_abuseipdb(value)
        risk = max(risk, abuse["abuse_confidence_score"])
    verdict = compute_verdict(risk)
    return {
        "value": value,
        "ioc_type": ioc_type,
        "risk_score": risk,
        "verdict": verdict,
        "recommendation": "block" if verdict == "malicious" else "monitor",
        "virustotal": vt,
        "abuseipdb": abuse,
        "enrichment_mode": "simulated",
    }

print("Funciones cargadas OK")

## 2. Prueba de detección de tipos

In [ ]:
muestras = [
    "8.8.8.8",
    "203.0.113.50",
    "192.168.1.100",
    "d41d8cd98f00b204e9800998ecf8427e",  # md5
    "da39a3ee5e6b4b0d3255bfef95601890afd80709",  # sha1
    "http://malicious.example/payload",
    "no-es-un-ioc",
]

df_tipos = pd.DataFrame(
    [{"valor": v, "tipo": detect_ioc_type(v)} for v in muestras]
)
display(df_tipos)

fig, ax = plt.subplots(figsize=(6, 3))
df_tipos["tipo"].value_counts().plot(kind="bar", ax=ax, color="#0ea5e9")
ax.set_title("Distribución de tipos IOC (muestra)")
ax.set_ylabel("Cantidad")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Enriquecer un lote de IPs y ver métricas

Umbrales (como en `ioc_service.py`):
- `risk_score >= 75` → **malicious** → recomendación `block`
- `>= 40` → **suspicious**
- resto → **clean**

In [ ]:
ips_lab = [
    "8.8.8.8",
    "1.1.1.1",
    "203.0.113.10",
    "203.0.113.50",
    "198.51.100.42",
    "192.0.2.55",
    "10.0.0.5",
    "172.16.0.8",
    "185.220.101.1",
    "45.33.32.156",
]

resultados = [enrich_ioc(ip) for ip in ips_lab]
df = pd.DataFrame(
    [
        {
            "ip": r["value"],
            "risk_score": r["risk_score"],
            "verdict": r["verdict"],
            "recommendation": r["recommendation"],
            "vt_malicious": r["virustotal"]["malicious"],
            "abuse_score": r["abuseipdb"]["abuse_confidence_score"] if r["abuseipdb"] else None,
        }
        for r in resultados
    ]
)
display(df)

print("\n--- Métricas de triaje ---")
print(f"IPs analizadas:     {len(df)}")
print(f"Media risk_score:   {df['risk_score'].mean():.1f}")
print(f"Mediana risk_score: {df['risk_score'].median():.1f}")
print(f"A bloquear:         {(df['recommendation'] == 'block').sum()}")
print(f"A monitorizar:      {(df['recommendation'] == 'monitor').sum()}")
print("\nVeredictos:")
print(df["verdict"].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = {"malicious": "#e11d48", "suspicious": "#f59e0b", "clean": "#10b981"}
bar_colors = [colors[v] for v in df["verdict"]]
axes[0].bar(df["ip"], df["risk_score"], color=bar_colors)
axes[0].axhline(75, color="#e11d48", linestyle="--", linewidth=1, label="umbral malicious")
axes[0].axhline(40, color="#f59e0b", linestyle="--", linewidth=1, label="umbral suspicious")
axes[0].set_title("Risk score por IP")
axes[0].set_ylabel("risk_score (0-100)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8)

df["verdict"].value_counts().reindex(["malicious", "suspicious", "clean"]).fillna(0).plot(
    kind="pie",
    ax=axes[1],
    colors=[colors["malicious"], colors["suspicious"], colors["clean"]],
    autopct="%1.0f%%",
)
axes[1].set_ylabel("")
axes[1].set_title("Distribución de veredictos")

plt.tight_layout()
plt.show()

## 4. Desglose VirusTotal simulado (una IP)

El score VT pondera motores: `malicious*100 + suspicious*50` / total.

In [ ]:
ejemplo = enrich_ioc("203.0.113.50")
vt = ejemplo["virustotal"]
print("Resultado completo:")
for k, v in ejemplo.items():
    if k not in ("virustotal", "abuseipdb"):
        print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(6, 3))
labels = ["malicious", "suspicious", "harmless", "undetected"]
vals = [vt[l] for l in labels]
ax.bar(labels, vals, color=["#e11d48", "#f59e0b", "#10b981", "#64748b"])
ax.set_title(f"VirusTotal simulado — {ejemplo['value']}")
ax.set_ylabel("Motores")
plt.tight_layout()
plt.show()

## 5. (Opcional) Importar el módulo real del backend

Si tienes el repo en disco, puedes importar `ioc_enrichment` sin arrancar Flask:

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "formacion":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

from app.services.ioc_enrichment import detect_ioc_type as real_detect
from app.services.ioc_enrichment import simulate_abuseipdb as real_abuse

ip = "198.51.100.42"
print("Tipo (módulo real):", real_detect(ip))
print("AbuseIPDB simulado (módulo real):", real_abuse(ip))
assert real_detect(ip) == detect_ioc_type(ip)
print("✓ Coincide con la copia pedagógica del notebook")

**Siguiente:** `03_pipeline_alerta_respuesta.ipynb` — de la alerta webhook al playbook.